# 🧊 Cenário de Dados: Sistema de Despachos (Apache Iceberg)

Vamos replicar o nosso cenário de logística utilizando o formato Apache Iceberg. O Iceberg é excelente para lidar com grandes volumes de dados e evoluções complexas de tabelas, mantendo a integridade das nossas transações (corridas e motoristas).

## Modelo Entidade-Relacionamento (ER)
```mermaid
erDiagram
    MOTORISTAS ||--o{ CORRIDAS : "realiza"
    
    MOTORISTAS {
        int id_motorista PK
        string nome
        string veiculo
        string status
    }
    
    CORRIDAS {
        int id_corrida PK
        int id_motorista FK
        string origem
        string destino
        float valor_corrida
        string status_corrida
    }

In [1]:
import os
import pyspark

# =====================================================================
# 1. Configurando o Spark para usar o Apache Iceberg
# =====================================================================
hadoop_home = os.path.abspath("hadoop_win")
os.environ["HADOOP_HOME"] = hadoop_home
os.environ["PATH"] += os.pathsep + os.path.join(hadoop_home, "bin")

print("Ligando o motor do Apache Iceberg...")

builder = pyspark.sql.SparkSession.builder.appName("Iceberg_Despachos") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.meu_catalogo", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.meu_catalogo.type", "hadoop") \
    .config("spark.sql.catalog.meu_catalogo.warehouse", os.path.abspath("iceberg_warehouse")) \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0")

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("✅ Spark com Iceberg iniciado com sucesso!\n")

# =====================================================================
# 2. Comandos DDL: Criando Banco de Dados e Tabelas
# =====================================================================
spark.sql("CREATE NAMESPACE IF NOT EXISTS meu_catalogo.despachos")

spark.sql("""
CREATE TABLE IF NOT EXISTS meu_catalogo.despachos.motoristas (
    id_motorista INT,
    nome STRING,
    veiculo STRING,
    status STRING
) USING iceberg
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS meu_catalogo.despachos.corridas (
    id_corrida INT,
    id_motorista INT,
    origem STRING,
    destino STRING,
    valor_corrida FLOAT,
    status_corrida STRING
) USING iceberg
""")

print("✅ Tabelas criadas no formato Iceberg!\n")

# =====================================================================
# 3. Operações: INSERT, UPDATE e DELETE
# =====================================================================
print("🚀 Iniciando operações no Iceberg...\n")

print("1️⃣ INSERT: Adicionando motoristas...")
spark.sql("""
INSERT INTO meu_catalogo.despachos.motoristas VALUES 
(1, 'João Silva', 'Fiat Uno', 'Ativo'),
(2, 'Maria Souza', 'Chevrolet Onix', 'Inativo')
""")
spark.sql("SELECT * FROM meu_catalogo.despachos.motoristas").show()

print("2️⃣ UPDATE: Maria comprou um carro novo e ficou ativa...")
spark.sql("""
UPDATE meu_catalogo.despachos.motoristas 
SET veiculo = 'Honda Civic', status = 'Ativo'
WHERE id_motorista = 2
""")
spark.sql("SELECT * FROM meu_catalogo.despachos.motoristas").show()

print("3️⃣ DELETE: Removendo o João Silva...")
spark.sql("""
DELETE FROM meu_catalogo.despachos.motoristas 
WHERE id_motorista = 1
""")
spark.sql("SELECT * FROM meu_catalogo.despachos.motoristas").show()

Ligando o motor do Apache Iceberg...
✅ Spark com Iceberg iniciado com sucesso!

✅ Tabelas criadas no formato Iceberg!

🚀 Iniciando operações no Iceberg...

1️⃣ INSERT: Adicionando motoristas...
+------------+-----------+--------------+-------+
|id_motorista|       nome|       veiculo| status|
+------------+-----------+--------------+-------+
|           1| João Silva|      Fiat Uno|  Ativo|
|           2|Maria Souza|Chevrolet Onix|Inativo|
+------------+-----------+--------------+-------+

2️⃣ UPDATE: Maria comprou um carro novo e ficou ativa...
+------------+-----------+-----------+------+
|id_motorista|       nome|    veiculo|status|
+------------+-----------+-----------+------+
|           2|Maria Souza|Honda Civic| Ativo|
|           1| João Silva|   Fiat Uno| Ativo|
+------------+-----------+-----------+------+

3️⃣ DELETE: Removendo o João Silva...
+------------+-----------+-----------+------+
|id_motorista|       nome|    veiculo|status|
+------------+-----------+-----------+---